# Bootstrap legacy Python projects

From the first chapter, we know that Python has been around for more than three decades and evolves continously. Today, most modern projects use a ``pyproject.toml`` file. However, many long-lived projects were created long before and still rely on familiar tools such as ``pip``, ``requirements.txt``, ``setup.py``, or ``setup.cfg``. Upgrading or refreshing always introduces some level of risk.

> *It works in production. Don't touch it.*

Now imagine it's your first day at a new company. Your team lead walks over to your desk and says:

> "We've got a small internal web service that needs to be tested from time to time. Nothing fancy—just make sure the endpoints are still responding."

Sounds simple enough.

## Start the server

In [43]:
!rm -rf $HOME/tmp/bobs-server/ && mkdir -p $HOME/tmp/bobs-server/ && cp -r $HOME/repos/ValentinTwin1206/modern-python-devops-egineering/projects/projXY_bobs_webserver/* $HOME/tmp/bobs-server/

In [20]:
!cd $HOME/tmp/bobs-server/server && python3 -m venv myvenv && ls -la $HOME/tmp/bobs-server/server

total 40
drwxr-xr-x 5 fixcfhu fixcfhu 4096 Aug  4 07:50 .
drwxr-xr-x 5 fixcfhu fixcfhu 4096 Aug  4 07:50 ..
drwxr-xr-x 2 fixcfhu fixcfhu 4096 Aug  4 07:50 .ipynb_checkpoints
drwxr-xr-x 2 fixcfhu fixcfhu 4096 Aug  4 07:50 files
-rw-r--r-- 1 fixcfhu fixcfhu 9208 Aug  4 07:50 main.py
drwxr-xr-x 5 fixcfhu fixcfhu 4096 Aug  4 07:50 myvenv
-rw-r--r-- 1 fixcfhu fixcfhu  137 Aug  4 07:50 requirements.txt
-rw-r--r-- 1 fixcfhu fixcfhu  478 Aug  4 07:50 setup.py


In [21]:
!cd $HOME/tmp/bobs-server/server && myvenv/bin/python3 -m pip install -r requirements.txt

  Using cached requests-2.0.0-py2.py3-none-any.whl.metadata (28 kB)
  Using cached urllib3-1.7.1-py3-none-any.whl
  Using cached certifi-2015.04.28-py2.py3-none-any.whl.metadata (1.5 kB)
  Using cached chardet-2.1.1-py3-none-any.whl
Using cached requests-2.0.0-py2.py3-none-any.whl (391 kB)
Using cached certifi-2015.04.28-py2.py3-none-any.whl (373 kB)


In [22]:
!cd $HOME/tmp/bobs-server/server && myvenv/bin/python3 main.py

Traceback (most recent call last):
  File "/home/fixcfhu/tmp/bobs-server/server/main.py", line 10, in <module>
    import imp  
    ^^^^^^^^^^
ModuleNotFoundError: No module named 'imp'


##### Conclusion

Setting up the server locally is not that easy because the server itselfs rely on ``Python3.9`` and is using libraries that are not compatible with our system wide installed Python.

What options do we have?
* seperating each project with their own oci (docker) containers
* installing Python3.9 on the system

## Setup Python 3.9

There are multiple ways how to setup Python on a running system. One option is to install it via an ppa remote. It supports the installation of multiple Python versions side-by-side on the system. However the installation at least requires some basic system knowledge. 

The official [instructions](https://launchpad.net/~deadsnakes/+archive/ubuntu/ppa) are offering the following guide

```
This PPA can be added to your system manually by copying the lines below and adding them to your system's software sources.

Display sources.list entries for: 
Noble (24.04)
deb https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble main 
deb-src https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble main 
Signing key:
4096R/F23C5A6CF475977595C89F51BA6932366A755776 (What is this?)
Fingerprint:
F23C5A6CF475977595C89F51BA6932366A755776
```

```bash
# get the signing key and convert it in apt's keyring binary format
curl -fsSL "https://keyserver.ubuntu.com/pks/lookup?op=get&search=0xF23C5A6CF475977595C89F51BA6932366A755776" | gpg --dearmor | sudo tee /etc/apt/keyrings/deadsnakes.gpg >/dev/null
```

```bash
# setup the remotes
cat <<EOF | sudo tee /etc/apt/sources.list.d/deadsnakes.list
deb [trusted=yes signed-by=/etc/apt/keyrings/deadsnakes.gpg] https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble main
deb-src [trusted=yes signed-by=/etc/apt/keyrings/deadsnakes.gpg] https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble main
EOF
```

```bash
# install python3.9
apt install python3.9 python3.9-venv python3.9-pip
```

In [ ]:
!cd $HOME/tmp/bobs-server/ && rm -f myvenv && python3.9 -m venv myvenv && myvenv/bin/python3 -m pip install -r requirements.txt

In [ ]:
!cd $HOME/tmp/bobs-server/ && myvenv/bin/python3 main.py

---

Now let's try to pin the Python interpreter to version 3.9

---

In [ ]:
!cd $HOME/tmp/demo-project && uv python pin 3.9 --verbose

##### Conclusion

Installing Python3.9 system-wide and spawning the server through a virtual environment at least work. However, we have seen that the installation is a bit tricky and requires basic system knowledge. It also seems a bit intransparent, and it is also loosly coupled from the server.

## Setup a Docker container

We started to setup the old web server from bob and quickly noticed that the setup is limited and crashes our system configuration. After installing Python 3.9 we finally managed to fully bootstrap the server. Moreover, we manipulated our system with the new Python installtion, nothing serious but in reality we are trying to hardly install and change anything on the system, that is when OCI containers comes into play, because they give us the possibility to provide application in an isolated way. 

We are now going to provide Bob's server inside an oci container and we are going to use ``docker`` to build the image. To fully provide Bob's server, we need to realize the following:
* copying all source artifacts in the image
* setting up Python 3.9
* creating a new user on the system *(starting a container as root is not good practise)*
* installing the needed Python requirements
* defining an entrypoint 

In [44]:
!cd $HOME/tmp/bobs-server/ && docker build -f Dockerfile -t bobs-webserver:2.4.13 -q .

sha256:c774182810e701a20a7c36dc440b9283cd7824474a219f4d4621198a842ffb49


In [ ]:
!docker run --rm -it bobs-webserver:2.4.13

/app/server/main.py:10: DeprecationWarning: the imp module is deprecated in favour of importlib; see the module's documentation for alternative uses
  import imp
/app/server/main.py:58: DeprecationWarning: distutils Version classes are deprecated. Use packaging.version instead.
  if LooseVersion(platform.python_version()) >= LooseVersion("3.9"):
Using modern compatibility mode.
Legacy API Service
Python: 3.9.25 (main, Nov  7 2025, 18:07:57) 
[GCC 13.3.0]
Server: legacy-api
Build: 1837
Compatibility: 7
Listening on http://127.0.0.1:8000



>NOTE: As an alternative we can also use a predefined Python image from [DockerHub]() e.g. the hardened [Python3.9](https://hub.docker.com/hardened-images/catalog/dhi/python/images/python%2Fdebian-13%2F3.9/sha256-2bc15a417cad8b0eb542fdff090536a8efd2d54b72d531d585f559054b054f86) image. 